# Adım 3: Spark Structured Streaming + Delta Lake

Bu notebook'ta:
- Kafka'dan gelen loan-events akışı okunur
- **Bronze** katmanına ham veri yazılır
- **Silver** katmanında veri temizlenir
- **Gold** katmanında analiz için hazır tablo oluşturulur

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, when, regexp_replace, trim, count, isnan
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

JARS_DIR = '/app/jars'
DELTA_BASE = os.environ.get('DELTA_BASE', '/app/delta_lake')

jars = ','.join(
    f'{JARS_DIR}/{f}' for f in os.listdir(JARS_DIR) if f.endswith('.jar')
)

spark = (
    SparkSession.builder
    .appName('CreditRisk-Step3')
    .master('local[*]')
    .config('spark.jars', jars)
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark Version:', spark.version)
print('Delta Base:', DELTA_BASE)

26/05/13 11:10:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 11:10:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/13 11:10:06 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark Version: 3.5.8
Delta Base: /app/delta_lake


## Şema Tanımı

In [2]:
LOAN_SCHEMA = StructType([
    StructField("timestamp",           StringType()),
    StructField("loan_id",             StringType()),
    StructField("event_type",          StringType()),
    StructField("loan_amnt",           DoubleType()),
    StructField("funded_amnt",         DoubleType()),
    StructField("term",                StringType()),
    StructField("int_rate",            StringType()),
    StructField("installment",         DoubleType()),
    StructField("grade",               StringType()),
    StructField("sub_grade",           StringType()),
    StructField("emp_length",          StringType()),
    StructField("home_ownership",      StringType()),
    StructField("annual_inc",          DoubleType()),
    StructField("verification_status", StringType()),
    StructField("issue_d",             StringType()),
    StructField("loan_status",         StringType()),
    StructField("purpose",             StringType()),
    StructField("dti",                 DoubleType()),
    StructField("delinq_2yrs",         DoubleType()),
    StructField("open_acc",            DoubleType()),
    StructField("pub_rec",             DoubleType()),
    StructField("revol_bal",           DoubleType()),
    StructField("revol_util",          StringType()),
    StructField("total_acc",           DoubleType()),
])

## Kafka'dan Okuma & Bronze Katmanı

In [3]:
# Streaming islemi 'make streaming' komutuyla Docker'da calistirildi
# Bronze verisi zaten Delta Lake'e yazildi - dogrudan okuyoruz

bronze_path = f'{DELTA_BASE}/bronze/loans'
silver_path = f'{DELTA_BASE}/silver/loans'
gold_path   = f'{DELTA_BASE}/gold/loans'

for name, path in [('Bronze', bronze_path), ('Silver', silver_path), ('Gold', gold_path)]:
    exists = os.path.exists(path)
    print(f'{name:6s}: {"MEVCUT" if exists else "EKSIK"} -> {path}')

print()
bronze_df = spark.read.format('delta').load(bronze_path)
print(f'Bronze kayit sayisi: {bronze_df.count():,}')

Bronze: MEVCUT -> /app/delta_lake/bronze/loans
Silver: MEVCUT -> /app/delta_lake/silver/loans
Gold  : MEVCUT -> /app/delta_lake/gold/loans



26/05/13 11:10:23 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/05/13 11:10:24 ERROR NonFateSharingFuture: Failed to get result from future
scala.runtime.NonLocalReturnControl
26/05/13 11:10:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 5:=======================================================> (49 + 1) / 50]

Bronze kayit sayisi: 127,072


## Silver Katmanı — Veri Temizleme

In [4]:
bronze_df = spark.read.format("delta").load(f"{DELTA_BASE}/bronze/loans")
print(f"Bronze kayıt sayısı: {bronze_df.count():,}")
bronze_df.printSchema()

Bronze kayıt sayısı: 127,629
root
 |-- timestamp: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: double (nullable = true)
 |-- open_acc: double (nullable = true)
 |-- pub_rec: double (nullable = true)
 |-- revol_bal: double (nullable = true)
 |-- revol_util: string (nullable = true)
 |-- total_acc: doubl

In [5]:
# Null analizi
null_counts = bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in bronze_df.columns
])
null_counts.show()

[Stage 25:===>                                                    (1 + 15) / 16]

+---------+-------+----------+---------+-----------+----+--------+-----------+-----+---------+----------+--------------+----------+-------------------+-------+-----------+-------+---+-----------+--------+-------+---------+----------+---------+
|timestamp|loan_id|event_type|loan_amnt|funded_amnt|term|int_rate|installment|grade|sub_grade|emp_length|home_ownership|annual_inc|verification_status|issue_d|loan_status|purpose|dti|delinq_2yrs|open_acc|pub_rec|revol_bal|revol_util|total_acc|
+---------+-------+----------+---------+-----------+----+--------+-----------+-----+---------+----------+--------------+----------+-------------------+-------+-----------+-------+---+-----------+--------+-------+---------+----------+---------+
|        0|      0|         0|        0|          0|   0|       0|          0|    0|        0|         0|             0|         0|                  0|      0|          0|      0|  0|          0|       0|      0|        0|         0|        0|
+---------+-------+-----

In [6]:
silver_df = (
    bronze_df
    .filter(col("loan_id").isNotNull())
    .filter(col("loan_amnt") > 0)
    .filter(col("annual_inc") > 0)
    .filter(col("loan_status").isNotNull())
    .dropDuplicates(["loan_id"])
    .withColumn("int_rate_pct",
                regexp_replace(trim(col("int_rate")), "%", "").cast("double"))
    .withColumn("revol_util_pct",
                regexp_replace(trim(col("revol_util")), "%", "").cast("double"))
    .withColumn("event_time", to_timestamp(col("timestamp")))
    .withColumn("is_default", when(
        col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1
    ).otherwise(0))
    .drop("int_rate", "revol_util", "timestamp")
)

print(f"Silver kayıt sayısı: {silver_df.count():,}")
print(f"Temerrüt oranı: {silver_df.filter(col('is_default')==1).count() / silver_df.count():.2%}")

silver_df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/silver/loans")
print("Silver katmanına yazıldı.")

Silver kayıt sayısı: 64,985


Temerrüt oranı: 18.49%


26/05/13 11:11:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/13 11:11:19 ERROR NonFateSharingFuture: Failed to get result from future  
scala.runtime.NonLocalReturnControl
                                                                                

Py4JJavaError: An error occurred while calling o216.save.
: io.delta.exceptions.ConcurrentAppendException: Files were added to the root of the table by a concurrent update. Please try the operation again.
Conflicting commit: {"timestamp":1778670673099,"operation":"STREAMING UPDATE","operationParameters":{"outputMode":Append,"queryId":10b98481-29d8-49d1-ab1c-e0190778062e,"epochId":92},"readVersion":97,"isolationLevel":"Serializable","isBlindAppend":true,"operationMetrics":{"numRemovedFiles":"0","numOutputRows":"929","numOutputBytes":"1550394","numAddedFiles":"199"},"engineInfo":"Apache-Spark/3.5.8 Delta-Lake/3.0.0","txnId":"71d25940-e162-4ffa-8ab6-c25196ab2a4c"}
Refer to https://docs.delta.io/latest/concurrency-control.html for more details.
	at org.apache.spark.sql.delta.DeltaErrorsBase.concurrentAppendException(DeltaErrors.scala:2293)
	at org.apache.spark.sql.delta.DeltaErrorsBase.concurrentAppendException$(DeltaErrors.scala:2284)
	at org.apache.spark.sql.delta.DeltaErrors$.concurrent

## Gold Katmanı — Analiz için Hazır Tablo

In [ ]:
from pyspark.sql.functions import year, month

gold_df = (
    silver_df
    .filter(col("loan_status").isin(
        "Fully Paid", "Charged Off", "Default", "Late (31-120 days)"
    ))
    .withColumn("issue_year",  year(to_timestamp(col("issue_d"), "MMM-yyyy")))
    .withColumn("issue_month", month(to_timestamp(col("issue_d"), "MMM-yyyy")))
    .withColumn("term_months",
                regexp_replace(trim(col("term")), " months", "").cast("integer"))
)

print(f"Gold kayıt sayısı: {gold_df.count():,}")
gold_df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/gold/loans")
print("Gold katmanına yazıldı.")

## Delta Lake Doğrulama

In [ ]:
gold = spark.read.format("delta").load(f"{DELTA_BASE}/gold/loans")
print("Gold tablo şeması:")
gold.printSchema()
gold.show(5)

In [ ]:
gold.groupBy("loan_status").count().orderBy("count", ascending=False).show()